In [ ]:
# ─── CELL 1: ENVIRONMENT SETUP ────────────────────────────────────────
# Run this cell first to install required packages in Colab
!pip install -q "numpy<2.0.0" vectorbt yfinance fredapi statsmodels scipy numba openpyxl requests

: 

In [ ]:
# ─── CELL 2: IMPORTS & CONFIGURATION ──────────────────────────────────
import os, sys, warnings, requests, json
from datetime import datetime, timedelta
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
warnings.filterwarnings("ignore")
import numba
from numba import jit, prange
import vectorbt as vbt
import yfinance as yf
from fredapi import Fred
import scipy.stats as stats

print(f"Python {sys.version}")
print(f"NumPy {np.__version__} | pandas {pd.__version__} | vectorbt {vbt.__version__}")

# ─── CONFIGURATION ───
START_DATE   = "2019-01-01"
END_DATE     = datetime.today().strftime("%Y-%m-%d")

# => USER: PASTE YOUR API KEYS HERE <=
EIA_API_KEY  = "YOUR_EIA_API_KEY_HERE"
FRED_API_KEY = "YOUR_FRED_API_KEY_HERE"

CP_HORMUZ    = "chokepoint6"
CP_BAB       = "chokepoint2"
CP_SUEZ      = "chokepoint1"
CP_CAPE      = "chokepoint14"


In [ ]:
# ─── CELL 3: DATA INGESTION (PORTWATCH & FINANCE) ──────────────────────
import os
csv_filename = 'portwatch_chokepoints.csv'

if not os.path.exists(csv_filename):
    print("WARNING: 'portwatch_chokepoints.csv' not found. Please upload it!")

print("Loading PortWatch CSV...")
df_pw = pd.read_csv(csv_filename)
if 'date' not in df_pw.columns and 'Date' in df_pw.columns:
    df_pw.rename(columns={'Date': 'date'}, inplace=True)
df_pw['date'] = pd.to_datetime(df_pw['date']).dt.tz_localize(None)

chokepoints = [CP_HORMUZ, CP_BAB, CP_SUEZ, CP_CAPE]
dfs = []
for cp in chokepoints:
    d = df_pw[df_pw['portid'] == cp].copy()
    d.set_index('date', inplace=True)
    d = d.sort_index()
    # Ensure we use Tanker traffic for Oil modeling
    call_col = 'n_tanker' if 'n_tanker' in d.columns else 'n_cargo'
    series = d[call_col]
    series.name = cp
    dfs.append(series)

df_ships = pd.concat(dfs, axis=1)
df_ships = df_ships.resample('D').sum().fillna(0)

# Critical Fix: 7-day rolling mean to capture short-term volatility spikes!
df_ships_7d = df_ships.rolling(window=7, min_periods=1).mean()

for cp in chokepoints:
    mean_7d = df_ships_7d[cp].expanding().mean()
    std_7d = df_ships_7d[cp].expanding().std()
    df_ships[f'{cp}_z'] = (df_ships_7d[cp] - mean_7d) / (std_7d + 1e-9)

df_ships = df_ships.dropna()

print("Downloading Brent Crude Data...")
oil = yf.download("BZ=F", start=START_DATE, end=END_DATE, progress=False)
df_oil = pd.DataFrame()
df_oil['brent_close'] = oil['Close']
df_oil['brent_high'] = oil['High']
df_oil['brent_low'] = oil['Low']
df_oil.index = pd.to_datetime(df_oil.index).tz_localize(None)
df_oil = df_oil.dropna()

print("Downloading VIX Data...")
fred = Fred(api_key=FRED_API_KEY)
vix = fred.get_series('VIXCLS', start_date=START_DATE)
df_vix = pd.DataFrame({'vix': vix})
df_vix.index = pd.to_datetime(df_vix.index).tz_localize(None)
df_vix = df_vix.dropna()

df_master = df_oil.join(df_vix, how='outer')
df_master['vix'] = df_master['vix'].ffill()
df_master = df_master.dropna()

df_master = df_master.join(df_ships[[f'{c}_z' for c in chokepoints]], how='left')
df_master.ffill(inplace=True)
df_master.dropna(inplace=True)

df_master['sma_50'] = df_master['brent_close'].rolling(50).mean()

delta = df_master['brent_close'].diff()
gain = (delta.where(delta > 0, 0)).rolling(14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
rs = gain / (loss + 1e-9)
df_master['rsi'] = 100 - (100 / (1 + rs))

high_low = df_master['brent_high'] - df_master['brent_low']
high_close = np.abs(df_master['brent_high'] - df_master['brent_close'].shift())
low_close = np.abs(df_master['brent_low'] - df_master['brent_close'].shift())
tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
df_master['atr'] = tr.rolling(14).mean()

df_master.dropna(inplace=True)
print("Data fully cached and synchronized.")

In [ ]:
# ─── CELL 4: COMPOSITE DISRUPTION INDEX (CDI) & REGIME DETECTION ───────
df_master['vix_median_60'] = df_master['vix'].rolling(60).median()
df_master['vix_regime'] = (df_master['vix'] > df_master['vix_median_60']).astype(int)
df_master.dropna(inplace=True)

@jit(nopython=True)
def compute_cdi_numba(hormuz_z, bab_z, rerouting, persistence, vix_regime, w1=0.4, w2=0.3, w3=0.2, w4=0.1):
    n = len(hormuz_z)
    cdi = np.zeros(n)
    for i in range(n):
        # Allow negative values to capture relief spikes (shorts)
        h = min(max(-hormuz_z[i] / 4, -1.0), 1.0)
        b = min(max(-bab_z[i] / 4, -1.0), 1.0)
        r = rerouting[i]
        p = min(persistence[i] / 7.0, 1.0)
        raw = w1*h + w2*b + w3*r + w4*p
        cdi[i] = raw * vix_regime[i]
    return cdi

rerouting_proxy = np.where((df_master[f'{CP_BAB}_z'] < -1.5) & (df_master[f'{CP_CAPE}_z'] > 1.0), 1.0, 0.0)
persistence_proxy = np.where((df_master[f'{CP_HORMUZ}_z'].rolling(10).mean() < -1.0), 1.0, 0.0)

df_master['cdi'] = compute_cdi_numba(
    df_master[f'{CP_HORMUZ}_z'].values,
    df_master[f'{CP_BAB}_z'].values,
    rerouting_proxy,
    persistence_proxy,
    df_master['vix_regime'].fillna(0).values
)

df_master.fillna(0, inplace=True)
print("CDI Engine running. Data ready for optimization.")

In [ ]:
# ─── CELL 5: TRUE REGIME-ADJUSTED WALK-FORWARD OPTIMIZATION (NO LOOKAHEAD) ───
print("Running Dynamic Regime Detection WFO (Starting 2023)...\n")

years = sorted(df_master.index.year.unique())
wf_results = []
oos_returns_list = []

# Define the Regime Grid
T_grid = [0.10, 0.15, 0.20]
VIX_THRESH_grid = [20.0, 25.0, 30.0]
TP_ATR_grid = [1.5, 2.0]
LEVERAGE = 3.0

# Start WFO at 2023 (i=4) so 2019-2022 acts as the train base containing the Russia Shock.
for i in range(4, len(years)):
    test_year = years[i]
    test_start = f"{test_year}-01-01"
    test_end = f"{test_year}-12-31"
    train_end = f"{test_year-1}-12-31"
    
    df_train = df_master.loc[:train_end].copy()
    df_test = df_master.loc[test_start:test_end].copy()
    
    if len(df_test) < 10:
        continue
        
    print(f"Training on 2019-{test_year-1} -> Testing on {test_year}...")
    
    best_sharpe = -999
    best_params = {'T': 0.15, 'VIX_THRESH': 25.0, 'TP_ATR': 2.0}
    
    # --- GRID SEARCH ON TRAIN DATA ---
    for t_val in T_grid:
        for vix_thresh in VIX_THRESH_grid:
            for tp_atr in TP_ATR_grid:
                # 1. Filtered Momentum (Sniper)
                en_1 = (df_train['cdi'] > t_val) & (df_train['brent_close'] > df_train['sma_50']) & (df_train['rsi'] < 70)
                sen_1 = (df_train['cdi'] < -t_val) & (df_train['brent_close'] < df_train['sma_50']) & (df_train['rsi'] > 30)
                
                clean_en_1, clean_sen_1, ex_1, sex_1 = pd.Series(False, index=df_train.index), pd.Series(False, index=df_train.index), pd.Series(False, index=df_train.index), pd.Series(False, index=df_train.index)
                in_trade, in_short = False, False
                for j in range(len(en_1)):
                    current_price = df_train['brent_close'].iloc[j]
                    if en_1.iloc[j] and not in_trade and not in_short:
                        in_trade = True; clean_en_1.iloc[j] = True
                        tp_target = current_price + (tp_atr * df_train['atr'].iloc[j])
                    elif in_trade:
                        if df_train['cdi'].iloc[j] <= 0 or current_price >= tp_target:
                            ex_1.iloc[j] = True; in_trade = False
                            continue
                    if sen_1.iloc[j] and not in_short and not in_trade:
                        in_short = True; clean_sen_1.iloc[j] = True
                        tp_target = current_price - (tp_atr * df_train['atr'].iloc[j])
                    elif in_short:
                        if df_train['cdi'].iloc[j] >= 0 or current_price <= tp_target:
                            sex_1.iloc[j] = True; in_short = False
                            continue
                            
                sz_1 = pd.Series(LEVERAGE, index=df_train.index)
                pf_filt_train = vbt.Portfolio.from_signals(
                    close=df_train['brent_close'], entries=clean_en_1, exits=ex_1,
                    short_entries=clean_sen_1, short_exits=sex_1, sl_trail=0.03,
                    size=sz_1, size_type='percent', fees=0.0002, freq='D', direction='both',
                    upon_opposite_entry='ignore'
                )
                
                # 2. Continuous Trend (Machine Gun)
                raw_en = (df_train['cdi'] > t_val) & (df_train['brent_close'] > df_train['sma_50'])
                raw_sen = (df_train['cdi'] < -t_val) & (df_train['brent_close'] < df_train['sma_50'])
                safe_en = raw_en & ~raw_sen
                safe_sen = raw_sen & ~raw_en
                
                sz_2 = pd.Series(LEVERAGE, index=df_train.index)
                pf_cont_train = vbt.Portfolio.from_signals(
                    close=df_train['brent_close'], entries=safe_en, short_entries=safe_sen,
                    sl_trail=0.03, tp_stop=0.06, size=sz_2, size_type='percent',
                    upon_opposite_entry='ignore', fees=0.0002, freq='D', direction='both'
                )
                
                # 3. Dynamic Regime Allocation!
                w_filt = pd.Series(1.0, index=df_train.index)
                # When VIX hits crisis levels, shift 80% capital to Continuous Trend
                w_filt[df_train['vix'] >= vix_thresh] = 0.2  
                w_cont = 1.0 - w_filt
                
                comb_train_returns = (pf_filt_train.returns() * w_filt) + (pf_cont_train.returns() * w_cont)
                vbt_train = comb_train_returns.vbt.returns(freq='D')
                
                try: 
                    sr = vbt_train.sharpe_ratio()
                except: 
                    sr = 0
                if pd.isna(sr) or np.isinf(sr): 
                    sr = 0
                
                if sr > best_sharpe:
                    best_sharpe = sr
                    best_params = {'T': t_val, 'VIX_THRESH': vix_thresh, 'TP_ATR': tp_atr}

    print(f"   -> Found Optimal Params: {best_params} (Train Sharpe: {best_sharpe:.2f})")
    
    # --- EXECUTE ON OOS TEST DATA USING EXACT PARAMS ---
    t_val = best_params['T']
    vix_thresh = best_params['VIX_THRESH']
    tp_atr = best_params['TP_ATR']
    
    en_1 = (df_test['cdi'] > t_val) & (df_test['brent_close'] > df_test['sma_50']) & (df_test['rsi'] < 70)
    sen_1 = (df_test['cdi'] < -t_val) & (df_test['brent_close'] < df_test['sma_50']) & (df_test['rsi'] > 30)
    
    clean_en_1, clean_sen_1, ex_1, sex_1 = pd.Series(False, index=df_test.index), pd.Series(False, index=df_test.index), pd.Series(False, index=df_test.index), pd.Series(False, index=df_test.index)
    in_trade, in_short = False, False
    for j in range(len(en_1)):
        current_price = df_test['brent_close'].iloc[j]
        if en_1.iloc[j] and not in_trade and not in_short:
            in_trade = True; clean_en_1.iloc[j] = True
            tp_target = current_price + (tp_atr * df_test['atr'].iloc[j])
        elif in_trade:
            if df_test['cdi'].iloc[j] <= 0 or current_price >= tp_target:
                ex_1.iloc[j] = True; in_trade = False
                continue
        if sen_1.iloc[j] and not in_short and not in_trade:
            in_short = True; clean_sen_1.iloc[j] = True
            tp_target = current_price - (tp_atr * df_test['atr'].iloc[j])
        elif in_short:
            if df_test['cdi'].iloc[j] >= 0 or current_price <= tp_target:
                sex_1.iloc[j] = True; in_short = False
                continue
                
    sz_1 = pd.Series(LEVERAGE, index=df_test.index)
    pf_filt_test = vbt.Portfolio.from_signals(
        close=df_test['brent_close'], entries=clean_en_1, exits=ex_1,
        short_entries=clean_sen_1, short_exits=sex_1, sl_trail=0.03,
        size=sz_1, size_type='percent', fees=0.0002, freq='D', direction='both',
        upon_opposite_entry='ignore'
    )
    
    raw_en = (df_test['cdi'] > t_val) & (df_test['brent_close'] > df_test['sma_50'])
    raw_sen = (df_test['cdi'] < -t_val) & (df_test['brent_close'] < df_test['sma_50'])
    safe_en = raw_en & ~raw_sen
    safe_sen = raw_sen & ~raw_en
    
    sz_2 = pd.Series(LEVERAGE, index=df_test.index)
    pf_cont_test = vbt.Portfolio.from_signals(
        close=df_test['brent_close'], entries=safe_en, short_entries=safe_sen,
        sl_trail=0.03, tp_stop=0.06, size=sz_2, size_type='percent',
        upon_opposite_entry='ignore', fees=0.0002, freq='D', direction='both'
    )
    
    w_filt = pd.Series(1.0, index=df_test.index)
    w_filt[df_test['vix'] >= vix_thresh] = 0.2
    w_cont = 1.0 - w_filt
    
    comb_test_returns = (pf_filt_test.returns() * w_filt) + (pf_cont_test.returns() * w_cont)
    oos_returns_list.append(comb_test_returns)
    vbt_test = comb_test_returns.vbt.returns(freq='D')
    
    stats = vbt_test.stats()
    wf_results.append({
        'Test Year': test_year,
        'Best T': t_val,
        'Regime VIX': vix_thresh,
        'OOS Sharpe': stats.get('Sharpe Ratio', float('nan')),
        'OOS Return [%]': stats.get('Total Return [%]', float('nan')),
        'OOS Max DD [%]': stats.get('Max Drawdown [%]', float('nan'))
    })

print("\n=== TRUE REGIME-ADJUSTED OUT-OF-SAMPLE YEARLY PERFORMANCE ===")
display(pd.DataFrame(wf_results))

unified_oos_returns = pd.concat(oos_returns_list)
vbt_oos_unified = unified_oos_returns.vbt.returns(freq='D')
print("\n=== FINAL UNIFIED TRUE WFO REGIME OOS STATS (2023-2026) ===")
display(vbt_oos_unified.stats())
vbt_oos_unified.plot().show()

In [ ]:
# ─── CELL 6: DETAILED OOS PERFORMANCE PLOTS & PNL ────────────────────
import plotly.graph_objects as go

print("Visualizing True Out-Of-Sample Strategy Performance...")

# 1. Cumulative Returns (PnL)
fig_cum = unified_oos_returns.cumsum().vbt.plot(
    title="Cumulative Out-Of-Sample Returns (2023-2026)",
    yaxis_title="Cumulative Return",
    trace_kwargs=dict(name='Dynamic Regime Strategy')
)
fig_cum.show()

# 2. Drawdowns
print("\nVisualizing Drawdown Profile...")
fig_dd = vbt_oos_unified.drawdowns.plot()
fig_dd.update_layout(title="Out-Of-Sample Drawdown Profile")
fig_dd.show()

# 3. Rolling Sharpe Ratio (6-month)
print("\nVisualizing Rolling Stability (6-Month Sharpe)...")
rolling_sharpe = vbt_oos_unified.rolling_sharpe_ratio(window=126)
fig_rs = rolling_sharpe.vbt.plot(
    title="Rolling 6-Month Sharpe Ratio", 
    yaxis_title="Sharpe Ratio"
)
fig_rs.add_hline(y=1.0, line_dash="dash", line_color="red", annotation_text="1.0 Target")
fig_rs.show()
